In [17]:
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier


# Load Dataset

df = pd.read_csv("StudentPerformanceFactors.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")



# DEFINE TARGET (Adjust if needed)

# Replace 'exam_score' with your actual score column name if different
score_column = "exam_score"

if score_column not in df.columns:
    raise ValueError(f"'{score_column}' not found in dataset. Check column names.")

median_score = df[score_column].median()
df["risk"] = df[score_column].apply(lambda x: 1 if x < median_score else 0)

print("Median score used as threshold:", median_score)
print(df["risk"].value_counts())


# Drop leakage column

df = df.drop([score_column], axis=1)


# Encode categorical features

df = pd.get_dummies(df, drop_first=True)

X = df.drop("risk", axis=1)
y = df["risk"]


# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Train Model

model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)


# Evaluate

probs = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, model.predict(X_test)))


# Save Model

os.makedirs("model", exist_ok=True)

joblib.dump(model, "model/trained_model.pkl")
joblib.dump(X.columns.tolist(), "model/feature_columns.pkl")

print("Model trained successfully!")

Median score used as threshold: 67.0
risk
0    3725
1    2882
Name: count, dtype: int64
ROC-AUC: 0.9649261977597619
              precision    recall  f1-score   support

           0       0.89      0.93      0.91       745
           1       0.91      0.85      0.88       577

    accuracy                           0.90      1322
   macro avg       0.90      0.89      0.90      1322
weighted avg       0.90      0.90      0.90      1322

Model trained successfully!
